## RAG


In [27]:
import chromadb
from pypdf import PdfReader
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import os


In [28]:
# The .env lives at the repo root, the database and index next to this script
ROOT = Path.cwd().parent
PDF_PATH = ROOT / "data" / "rag_docs" / "irs_pub_523.pdf"

In [29]:
load_dotenv(ROOT / ".env")
BASE_URL = "https://openrouter.ai/api/v1"


In [32]:
client = OpenAI(base_url=BASE_URL)

# 1. Read PDF
reader = PdfReader(PDF_PATH)


In [34]:

text = ""

for page in reader.pages:
    text += page.extract_text() or ""

# 2. Split text into chunks
chunk_size = 500

chunks = [
    text[i:i + chunk_size]
    for i in range(0, len(text), chunk_size)
]

# 3. Create ChromaDB
chroma = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma.get_or_create_collection(
    name="pdf_documents"
)

# 4. Generate embeddings
for i, chunk in enumerate(chunks):

    embedding = client.embeddings.create(
        model="google/gemini-embedding-001",
        input=chunk
    ).data[0].embedding

    collection.upsert(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[embedding]
    )

print("PDF stored in ChromaDB!")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FirstChar': 65, '/FontDescriptor': IndirectObject(3640, 0, 1665315452752), '/BaseFont': '/KHAMKK+Helvetica-Black', '/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/LastChar': 85, '/Widths': [778, 0, 778, 0, 0, 0, 0, 0, 389, 0, 0, 0, 0, 833, 833, 0, 0, 0, 0, 722, 833]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FirstChar': 33, '/FontDescriptor': IndirectObject(3641, 0, 1665315452752), '/BaseFont': '/KHAMKL+Helvetica-Bold', '/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/LastChar': 33, '/Widths': [333]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FirstCh

PDF stored in ChromaDB!
